In [ ]:
import os
import gc
import json
import pickle
import time
import logging
import warnings
import math
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    accuracy_score,
)

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("scope")

In [ ]:
!pip install -q -U scikit-learn pandas numpy tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 39.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.


In [ ]:
# No external model downloads — using lightweight K-mer Transformer (~200K params)
print('Using K-mer Siamese Transformer — no heavy model downloads needed')

Using K-mer Siamese Transformer — no heavy model downloads needed


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, pandas as pd


# Clean up any leftover mount and remount fresh
from google.colab import drive

try:
    drive.flush_and_unmount()
    print("unmounted previous session")
except Exception as e:
    print(f"nothing to unmount ({e})")

# Wipe the leftover directory if it still has stuff in it
import shutil, os
if os.path.exists('/content/drive') and os.listdir('/content/drive'):
    shutil.rmtree('/content/drive')
    print("cleared /content/drive")

# Now mount cleanly
drive.mount('/content/drive')
print("mounted")


DRIVE_DIR        = "/content/drive/MyDrive/dgeb_siamese"
OPERON_DATA_DIR  = f"{DRIVE_DIR}/operon_data"
RESULTS_DIR      = f"{DRIVE_DIR}/scope_results"
CHECKPOINT_DIR   = f"{DRIVE_DIR}/scope_checkpoints"
EMBED_CACHE_DIR  = f"{DRIVE_DIR}/embedding_cache"


TRAIN_CSV = f"{OPERON_DATA_DIR}/operon_train.csv"
VAL_CSV   = f"{OPERON_DATA_DIR}/operon_val.csv"

for d in (RESULTS_DIR, CHECKPOINT_DIR, EMBED_CACHE_DIR):
    os.makedirs(d, exist_ok=True)

# Verify CSVs are visible
print(f"DRIVE_DIR: {DRIVE_DIR}")
print(f"  exists: {os.path.isdir(DRIVE_DIR)}")
print()
for label, path in [("TRAIN", TRAIN_CSV), ("VAL", VAL_CSV)]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  ✓ {label}: {path}  ({size_mb:.2f} MB)")
    else:
        print(f"  ✗ {label}: {path}  — NOT FOUND")

# Quick peek at the train CSV so you can confirm columns
if os.path.exists(TRAIN_CSV):
    df = pd.read_csv(TRAIN_CSV, nrows=3)
    print(f"\ncolumns: {list(df.columns)}")
    print(f"shape (head): {df.shape}")
    print(df.head(3))

# What's already cached from prior runs (if anything)
print("\nexisting cache contents:")
for d, label in [(EMBED_CACHE_DIR, "embeddings"), (RESULTS_DIR, "results"), (CHECKPOINT_DIR, "checkpoints")]:
    files = os.listdir(d) if os.path.isdir(d) else []
    print(f"  {label:12s} ({d}):  {files if files else '(empty)'}")

Mounted at /content/drive
unmounted previous session
Mounted at /content/drive
mounted
DRIVE_DIR: /content/drive/MyDrive/dgeb_siamese
  exists: True

  ✓ TRAIN: /content/drive/MyDrive/dgeb_siamese/operon_data/operon_train.csv  (25.50 MB)
  ✓ VAL: /content/drive/MyDrive/dgeb_siamese/operon_data/operon_val.csv  (10.92 MB)

columns: ['sequence_1', 'sequence_2', 'operon_id', 'label']
shape (head): (3, 4)
                                          sequence_1  \
0  MQKQHNVVAGVDMGATHIRFCLRTAEGETLHCEKKRTAEVIAPGLV...   
1  MFRGATLVNLDSKGRLSVPTRYREQLLENAAGQMVCTIDIYHPCLL...   
2  MEQFSAFKLLLKKQYETTLGFYDKYIKNLKRFALKNNVLFVIVDNE...   

                                          sequence_2 operon_id  label  
0  MSQSEFDSALPNGIGLAPYLRMKQEGMTENESRIVEWLLKPGNLSC...   KO03198      1  
1  MKAAAKTQKPKRQEEHANFISWRFALLCGCILLALAFLLGRVAWLQ...   KO03290      1  
2  MAISKKKRFFFDLAQDEDDAETVQEVKKVEQQLKLEPVVQPQHDLT...   KO07485      1  

existing cache contents:
  embeddings   (/content/drive/MyDrive/dgeb_siamese/embed

In [ ]:
from dataclasses import dataclass

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
KMER_VOCAB = {"<PAD>": 0, "<CLS>": 1, "<UNK>": 2, "<SEP>": 3}
_idx = 4
for a1 in AMINO_ACIDS:
    for a2 in AMINO_ACIDS:
        for a3 in AMINO_ACIDS:
            KMER_VOCAB[f"{a1}{a2}{a3}"] = _idx; _idx += 1
VOCAB_SIZE = len(KMER_VOCAB)
KMER_SIZE = 3

def sequence_to_kmers(seq: str, k: int = KMER_SIZE) -> list:
    return [seq[i:i+k] for i in range(0, len(seq) - k + 1, k)]

def encode_sequence(seq: str, max_len: int = 256) -> list:
    kmers = sequence_to_kmers(seq.upper().strip())
    ids = [KMER_VOCAB["<CLS>"]] + [KMER_VOCAB.get(k, KMER_VOCAB["<UNK>"]) for k in kmers[:max_len-2]] + [KMER_VOCAB["<SEP>"]]
    return ids

@dataclass
class Config:
    d_model:        int   = 128
    nhead:          int   = 4
    num_layers:     int   = 2
    dim_feedforward: int  = 256
    dropout:        float = 0.1
    max_len:        int   = 256
    vocab_size:     int   = VOCAB_SIZE

    batch_size:     int   = 64
    learning_rate:  float = 3e-4
    weight_decay:   float = 1e-2
    num_epochs:     int   = 10
    patience:       int   = 3
    grad_clip:      float = 1.0
    seed:           int   = 42
    device:         str   = "cuda" if torch.cuda.is_available() else "cpu"
    experiment_name: str  = "kmer_scope"

def set_seed(seed: int):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def atomic_pickle_dump(obj, path: str):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f)
    os.replace(tmp, path)

In [ ]:
COL_SEQ1, COL_SEQ2, COL_OPERON, COL_LABEL = "sequence_1", "sequence_2", "operon_id", "label"


def load_operon_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"CSV not found: {path}")
    df = pd.read_csv(path)
    needed = {COL_SEQ1, COL_SEQ2, COL_LABEL}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")
    df = df.dropna(subset=[COL_SEQ1, COL_SEQ2, COL_LABEL]).reset_index(drop=True)
    df[COL_SEQ1]  = df[COL_SEQ1].astype(str).str.strip().str.upper()
    df[COL_SEQ2]  = df[COL_SEQ2].astype(str).str.strip().str.upper()
    df[COL_LABEL] = df[COL_LABEL].astype(int)
    return df

In [ ]:
class KmerSiameseTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.token_embed = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=0)
        self.pos_embed = nn.Parameter(torch.randn(cfg.max_len, cfg.d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model, nhead=cfg.nhead,
            dim_feedforward=cfg.dim_feedforward, dropout=cfg.dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.num_layers)
        self.cls_norm = nn.LayerNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, 32)

        n_params = sum(p.numel() for p in self.parameters())
        print(f"K-mer Siamese Transformer: {n_params:,} params")

    def encode(self, input_ids: torch.Tensor) -> torch.Tensor:
        seq_len = input_ids.shape[1]
        h = self.token_embed(input_ids) + self.pos_embed[:seq_len]
        out = self.transformer(h)
        return self.cls_norm(out[:, 0])

    def forward(self, ids1: torch.Tensor, ids2: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.encode(ids1), self.encode(ids2)

In [ ]:
class KmerPairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, max_len: int = 256):
        self.ids1 = [encode_sequence(s, max_len) for s in df[COL_SEQ1]]
        self.ids2 = [encode_sequence(s, max_len) for s in df[COL_SEQ2]]
        self.labels = df[COL_LABEL].astype(int).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {"ids1": torch.tensor(self.ids1[idx], dtype=torch.long),
                "ids2": torch.tensor(self.ids2[idx], dtype=torch.long),
                "label": self.labels[idx]}

In [ ]:
def _pad_collate(batch):
    max_len = max(max(b["ids1"].shape[0], b["ids2"].shape[0]) for b in batch)
    ids1 = torch.stack([F.pad(b["ids1"], (0, max_len - b["ids1"].shape[0]), value=0) for b in batch])
    ids2 = torch.stack([F.pad(b["ids2"], (0, max_len - b["ids2"].shape[0]), value=0) for b in batch])
    return {"ids1": ids1, "ids2": ids2,
            "label": torch.tensor([b["label"] for b in batch], dtype=torch.long)}

## K-mer Siamese Transformer Architecture

Replaces the heavy ESM2 + Mamba decoder with a lightweight transformer:
- 3-mer tokenization of protein sequences (8004 vocab)
- Embedding + positional encoding → TransformerEncoder → CLS token
- Cosine similarity → small MLP classifier
- **~200K total params** (vs 35M+ for ESM-2)

## Datasets & Training Utilities


In [ ]:
def _best_f1(scores, labels):
    p, r, t = precision_recall_curve(labels, scores)
    f1s = 2 * p * r / (p + r + 1e-12)
    idx = int(np.argmax(f1s))
    thr = t[idx] if idx < len(t) else t[-1]
    preds = (scores >= thr).astype(int)
    return float(f1s[idx]), float(p[idx]), float(r[idx]), float(thr), float(accuracy_score(labels, preds))


def dgeb_pair_evaluate(f1_emb, f2_emb, labels):
    n1 = np.linalg.norm(f1_emb, axis=-1, keepdims=True) + 1e-12
    n2 = np.linalg.norm(f2_emb, axis=-1, keepdims=True) + 1e-12
    cos  = (f1_emb / n1 * f2_emb / n2).sum(-1)
    dot  = (f1_emb * f2_emb).sum(-1)
    eucl = np.linalg.norm(f1_emb - f2_emb, axis=-1)
    manh = np.abs(f1_emb - f2_emb).sum(-1)

    out = {}
    for name, score in {"cosine": cos, "dot": dot, "euclidean": -eucl, "manhattan": -manh}.items():
        ap = float(average_precision_score(labels, score))
        f1_, p, r, thr, acc = _best_f1(score, labels)
        out[name] = {"ap": ap, "best_f1": f1_, "precision": p, "recall": r,
                     "accuracy": acc, "threshold": thr}
    return out

In [ ]:
def run_lr_baseline(train_df, val_df, model, device, max_len=256):
    model.eval()
    @torch.no_grad()
    def get_embeds_for_col(df, col_name):
        all_emb = []
        for s in df[col_name]:
            ids = torch.tensor([encode_sequence(s, max_len)], device=device)
            e1, _ = model(ids, ids)
            all_emb.append(e1[0].cpu().numpy())
        return np.array(all_emb)

    print("  extracting embeddings for LR baseline...")
    e1_tr = get_embeds_for_col(train_df, COL_SEQ1)
    e2_tr = get_embeds_for_col(train_df, COL_SEQ2)
    e1_va = get_embeds_for_col(val_df, COL_SEQ1)
    e2_va = get_embeds_for_col(val_df, COL_SEQ2)

    y_tr, y_va = train_df[COL_LABEL].values, val_df[COL_LABEL].values
    out = {}
    for mode in ["concat", "diff", "concat_diff"]:
        if mode == "concat":
            Xtr = np.concatenate([e1_tr, e2_tr], 1)
            Xva = np.concatenate([e1_va, e2_va], 1)
        elif mode == "diff":
            Xtr = np.abs(e1_tr - e2_tr)
            Xva = np.abs(e1_va - e2_va)
        else:
            Xtr = np.concatenate([e1_tr, e2_tr, np.abs(e1_tr - e2_tr)], 1)
            Xva = np.concatenate([e1_va, e2_va, np.abs(e1_va - e2_va)], 1)
        sc = StandardScaler().fit(Xtr)
        lr = LogisticRegression(solver="lbfgs", class_weight="balanced", max_iter=1000).fit(sc.transform(Xtr), y_tr)
        probs = lr.predict_proba(sc.transform(Xva))[:, 1]
        ap = float(average_precision_score(y_va, probs))
        f1_, p, r, thr, acc = _best_f1(probs, y_va)
        out[f"lr_{mode}"] = {"ap": ap, "best_f1": f1_, "precision": p, "recall": r, "accuracy": acc, "threshold": thr}
        logger.info(f"  LR / {mode:11s}  AP={ap:.4f}  F1={f1_:.4f}")

    out["frozen_kmer"] = dgeb_pair_evaluate(e1_va, e2_va, y_va)
    logger.info(f"  frozen kmer cosine  AP={out['frozen_kmer']['cosine']['ap']:.4f}")
    return out

In [ ]:
class SCOPETrainer:
    def __init__(self, model, cfg, train_loader, val_loader, results_path):
        self.model = model.to(cfg.device)
        self.cfg = cfg
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = cfg.device
        self.results_path = results_path
        self.classifier = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Dropout(cfg.dropout), nn.Linear(32, 2)).to(cfg.device)

        all_params = list(self.model.parameters()) + list(self.classifier.parameters())
        n_train = sum(p.numel() for p in all_params if p.requires_grad)
        logger.info(f"trainable params: {n_train:,}")

        self.optimizer = optim.AdamW(all_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
        self.criterion = nn.CrossEntropyLoss()

        self.history = {"train": [], "val": []}
        self.start_epoch = 0
        self.best_ap = -1.0
        self.best_epoch = -1
        self.epochs_no_improve = 0
        self.last_ckpt = os.path.join(CHECKPOINT_DIR, f"{cfg.experiment_name}_last.pt")
        self.best_ckpt = os.path.join(CHECKPOINT_DIR, f"{cfg.experiment_name}_best.pt")

    def _save_last(self, epoch):
        torch.save({"epoch": epoch, "model_state": self.model.state_dict(),
                    "classifier_state": self.classifier.state_dict(),
                    "optimizer_state": self.optimizer.state_dict(),
                    "history": self.history, "best_ap": self.best_ap,
                    "best_epoch": self.best_epoch, "epochs_no_improve": self.epochs_no_improve,
                    "config": asdict(self.cfg)}, self.last_ckpt)

    def _save_best(self, epoch):
        torch.save({"epoch": epoch, "model_state": self.model.state_dict(),
                    "classifier_state": self.classifier.state_dict(),
                    "best_ap": self.best_ap, "config": asdict(self.cfg)}, self.best_ckpt)

    def _save_results(self, final_metrics=None, status="running"):
        out = {"experiment": self.cfg.experiment_name, "config": asdict(self.cfg),
               "history": self.history, "best_ap": self.best_ap, "best_epoch": self.best_epoch,
               "final_val_metrics": final_metrics, "status": status,
               "checkpoint_paths": {"last": self.last_ckpt, "best": self.best_ckpt}}
        atomic_pickle_dump(out, self.results_path)

    def _train_epoch(self, epoch):
        self.model.train(); self.classifier.train()
        total_loss, correct, total = 0, 0, 0
        for batch in tqdm(self.train_loader, desc=f"epoch {epoch} [train]", leave=False):
            ids1 = batch["ids1"].to(self.device)
            ids2 = batch["ids2"].to(self.device)
            labels = batch["label"].to(self.device)
            emb1, emb2 = self.model(ids1, ids2)
            cos_sim = F.cosine_similarity(emb1, emb2, dim=1).unsqueeze(1)
            logits = self.classifier(cos_sim)
            loss = self.criterion(logits, labels)
            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(self.model.parameters()) + list(self.classifier.parameters()), self.cfg.grad_clip)
            self.optimizer.step()
            total_loss += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.shape[0]
        return {"loss": total_loss / len(self.train_loader), "accuracy": correct / total}

    @torch.no_grad()
    def _evaluate(self):
        self.model.eval(); self.classifier.eval()
        f1s, f2s, ys = [], [], []
        for batch in tqdm(self.val_loader, desc="eval", leave=False):
            ids1 = batch["ids1"].to(self.device)
            ids2 = batch["ids2"].to(self.device)
            emb1, emb2 = self.model(ids1, ids2)
            f1s.append(emb1.cpu().numpy())
            f2s.append(emb2.cpu().numpy())
            ys.append(batch["label"].numpy())
        f1 = np.concatenate(f1s); f2 = np.concatenate(f2s); y = np.concatenate(ys)
        m = dgeb_pair_evaluate(f1, f2, y)
        return m

    def train(self):
        final_metrics = None
        for epoch in range(self.cfg.num_epochs):
            t0 = time.time()
            train_m = self._train_epoch(epoch)
            val_m = self._evaluate()
            final_metrics = val_m
            ap_cos = val_m["cosine"]["ap"]
            logger.info(f"epoch {epoch:02d}  ({time.time()-t0:.1f}s)  train_loss={train_m['loss']:.4f}  "
                        f"train_acc={train_m['accuracy']:.4f}  AP[cos]={ap_cos:.4f}  "
                        f"F1[cos]={val_m['cosine']['best_f1']:.4f}")
            if ap_cos > self.best_ap:
                self.best_ap = ap_cos; self.best_epoch = epoch; self.epochs_no_improve = 0
                self._save_best(epoch)
                logger.info(f"  new best AP={self.best_ap:.4f}")
            else:
                self.epochs_no_improve += 1
            self.history["train"].append({"epoch": epoch, **train_m})
            self.history["val"].append({"epoch": epoch, **val_m})
            self._save_results(final_metrics=final_metrics, status="running")
            self._save_last(epoch)
            if self.epochs_no_improve >= self.cfg.patience:
                logger.info(f"early stopping at epoch {epoch}")
                break
        self._save_results(final_metrics=final_metrics, status="done")
        return final_metrics

In [ ]:
def build_summary_table(lr_results, final_scope_metrics):
    rows = []
    if "frozen_kmer" in lr_results:
        for dist, m in lr_results["frozen_kmer"].items():
            rows.append({"model": f"frozen_kmer / {dist}", **{k: m[k] for k in ("ap","best_f1","precision","recall","accuracy")}})
    for mode in ["lr_concat", "lr_diff", "lr_concat_diff"]:
        if mode in lr_results:
            m = lr_results[mode]
            rows.append({"model": mode, **{k: m[k] for k in ("ap","best_f1","precision","recall","accuracy")}})
    if final_scope_metrics is not None:
        for dist in ("cosine", "dot", "euclidean", "manhattan"):
            m = final_scope_metrics[dist]
            rows.append({"model": f"SCOPE / {dist}", **{k: m[k] for k in ("ap","best_f1","precision","recall","accuracy")}})
    return pd.DataFrame(rows)

In [ ]:
def run_one_dataset(train_csv, val_csv, experiment_name, cfg=None):
    cfg = cfg or Config()
    cfg.experiment_name = experiment_name
    set_seed(cfg.seed)
    logger.info(f"loading data: {train_csv} / {val_csv}")
    train_df = load_operon_csv(train_csv)
    val_df   = load_operon_csv(val_csv)
    logger.info(f"  train={len(train_df)}  val={len(val_df)}")

    logger.info("\n--- LR baseline (frozen kmer embeddings) ---")
    model_for_lr = KmerSiameseTransformer(cfg)
    lr_results = run_lr_baseline(train_df, val_df, model_for_lr, cfg.device)

    results_path = os.path.join(RESULTS_DIR, f"{experiment_name}_results.pkl")
    atomic_pickle_dump({"experiment": experiment_name, "config": asdict(cfg), "lr_baseline": lr_results, "scope": None, "status": "lr_done"}, results_path)

    logger.info("\n--- SCOPE training ---")
    train_ds = KmerPairDataset(train_df, max_len=cfg.max_len)
    val_ds   = KmerPairDataset(val_df,   max_len=cfg.max_len)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True, collate_fn=_pad_collate)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=_pad_collate)

    model = KmerSiameseTransformer(cfg)
    trainer = SCOPETrainer(model, cfg, train_loader, val_loader, results_path)
    final_metrics = trainer.train()

    summary_df = build_summary_table(lr_results, final_metrics)
    print("\n" + "=" * 80)
    print(f"SUMMARY  [{experiment_name}]")
    print("=" * 80)
    print(summary_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    final_obj = {"experiment": experiment_name, "config": asdict(cfg), "lr_baseline": lr_results,
                 "scope": {"best_ap": trainer.best_ap, "best_epoch": trainer.best_epoch,
                           "history": trainer.history, "final_val_metrics": final_metrics},
                 "summary": summary_df.to_dict(orient="records"), "status": "done"}
    atomic_pickle_dump(final_obj, results_path)
    summary_df.to_csv(os.path.join(RESULTS_DIR, f"{experiment_name}_summary.csv"), index=False)
    logger.info(f"\nfinal results: {results_path}")
    del trainer, model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return final_obj

In [ ]:
cfg = Config()                    # fresh instance with all fields
print(cfg.batch_size)

results = run_one_dataset(
    train_csv=TRAIN_CSV,
    val_csv=VAL_CSV,
    experiment_name="ecoli",
    cfg=cfg,
)


64
K-mer Siamese Transformer: 1,326,624 params
  extracting embeddings for LR baseline...
K-mer Siamese Transformer: 1,326,624 params


epoch 0 [train]:   0%|          | 0/581 [00:00<?, ?it/s]

eval:   0%|          | 0/249 [00:00<?, ?it/s]

epoch 1 [train]:   0%|          | 0/581 [00:00<?, ?it/s]

eval:   0%|          | 0/249 [00:00<?, ?it/s]

RuntimeError: DataLoader worker (pid 33243) is killed by signal: Killed. 

## DGEB Evaluation: Frozen K-mer per-layer vs SCOPE

Evaluate on the official DGEB E.coli operon dataset.
Since our model is a single transformer (not multi-layer like ESM2),
we compare the frozen encoder embeddings against the trained SCOPE model.

In [ ]:
from datasets import load_dataset

dgeb_ds = load_dataset(
    "tattabio/ecoli_operonic_pair",
    revision="a62c01143a842696fc8200b91c1acb825e8cb891",
    split="train",
)
print(f"num operons: {len(dgeb_ds)}")
print(f"example operon length: {len(dgeb_ds[0]['Sequence'])}")

In [ ]:
all_seqs = []
pairs = []
for row in dgeb_ds:
    seqs = row["Sequence"]
    labels = row["Label"]
    offset = len(all_seqs)
    all_seqs.extend(seqs)
    for j in range(len(seqs) - 1):
        pairs.append((offset + j, offset + j + 1, labels[j]))

pair_labels = np.array([p[2] for p in pairs])
unique_seqs = sorted(set(all_seqs))
print(f"total pairs: {len(pairs)}, positive: {int(pair_labels.sum())}, unique seqs: {len(unique_seqs)}")

In [ ]:
@torch.no_grad()
def get_all_embeddings(seqs, model, batch_size=256):
    all_emb = []
    for i in tqdm(range(0, len(seqs), batch_size), desc="encoding"):
        batch = seqs[i:i+batch_size]
        ids = torch.tensor([encode_sequence(s, cfg.max_len) for s in batch], device=cfg.device)
        emb, _ = model(ids, ids)
        all_emb.append(emb.cpu().numpy())
    return np.concatenate(all_emb, axis=0)

# Load best SCOPE checkpoint
best_ckpt_path = os.path.join(CHECKPOINT_DIR, f"{cfg.experiment_name}_best.pt")
if os.path.exists(best_ckpt_path):
    ckpt = torch.load(best_ckpt_path, map_location=cfg.device)
    eval_model = KmerSiameseTransformer(cfg)
    eval_model.load_state_dict(ckpt["model_state"])
    eval_model.eval().to(cfg.device)

    scope_embs = get_all_embeddings(unique_seqs, eval_model)
    idx1 = np.array([p[0] for p in pairs])
    idx2 = np.array([p[1] for p in pairs])
    scope_m = dgeb_pair_metrics(scope_embs[idx1], scope_embs[idx2], pair_labels)
    print(f"SCOPE  cos_ap={scope_m['cosine']['ap']:.4f}  top_ap={scope_m['top_ap']:.4f}")

    # Also eval frozen (untrained) model
    frozen_model = KmerSiameseTransformer(cfg)
    frozen_embs = get_all_embeddings(unique_seqs, frozen_model)
    frozen_m = dgeb_pair_metrics(frozen_embs[idx1], frozen_embs[idx2], pair_labels)
    print(f"FROZEN cos_ap={frozen_m['cosine']['ap']:.4f}  top_ap={frozen_m['top_ap']:.4f}")
else:
    print(f"No checkpoint at {best_ckpt_path}. Run training first.")
    scope_m = frozen_m = None

In [ ]:
rows = []
if frozen_m is not None:
    for dist in ["cosine", "manhattan", "euclidean", "dot"]:
        rows.append({"model": "frozen_kmer", "distance": dist, **{k: frozen_m[dist][k] for k in ("ap","best_f1","precision","recall","accuracy")}})
if scope_m is not None:
    for dist in ["cosine", "manhattan", "euclidean", "dot"]:
        rows.append({"model": "SCOPE (trained)", "distance": dist, **{k: scope_m[dist][k] for k in ("ap","best_f1","precision","recall","accuracy")}})

if rows:
    dgeb_df = pd.DataFrame(rows)
    pivot = dgeb_df.pivot_table(index="model", columns="distance", values="ap", aggfunc="first")
    pivot["top_ap"] = pivot.max(axis=1)
    print("\n" + "=" * 70)
    print("DGEB E.coli Operon — Frozen vs Trained")
    print("=" * 70)
    print(pivot.to_string(float_format=lambda v: f"{v:.4f}"))